In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd
import os
from scipy.stats import gaussian_kde
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors
from matplotlib import colors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
lat = np.arange(-90, 91, 1)
lon = np.arange(0, 361, 1)

TC_path1d = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/genesis-new-1d/'
output_path = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TCT/'
TC_path2d = r'F:\study\data\noaa-20c\NOAA\NOAA_20C_TXT_30_all_bing\genesis-new-2d\\'
TCT_path = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/'
soda_path = r'F:\study\data\soda\\'
path_noaa = r'F:\study\data\noaa-20c\\'

In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2014-01')]
ep = ep[ep < np.datetime64('2014-01')]
mix = mix[mix < np.datetime64('2014-01')]
suc = suc[suc < np.datetime64('2014-01')]
events = [mix, ep, cp, suc]
# time = pd.date_range('1806-01', '2015-12', freq='MS')
# nt = len(time)
# nx, ny = len(lon), len(lat)
# ds_tct = xr.Dataset(
#     {
#         'tct':([], )
#     },
#     coords={
#         'time':(['time'], time),
#         'lat':(['lat'], lat),
#         'lon':(['lon'], lon)
#     }
# )
# tct_grid = np.zeros((nt, ny, nx))
tct = [[[] for i in range(24)] for j in range(3)]
for i in range(1, 2):
    file_name = TCT_path+f'merged_0{i:02d}.txt'
    df = pd.read_csv(file_name, header=None, sep=',', names=['lon', 'lat', 'intensity', 'time'], on_bad_lines='skip').dropna()
    times = df.loc[:, 'time'].to_numpy(dtype='datetime64[M]')
    LATS = df.loc[:, 'lat'].to_numpy(dtype='float32')
    LONS = df.loc[:, 'lon'].to_numpy(dtype='float32')
    ts = (times - np.datetime64('1806-01')).astype(int)
    ys = LATS + 90
    xs = LONS
    for j in range(3):
        event = events[j]
        for k in range(len(event)):
            for m in range(24):
                mask = times == event[k]+np.timedelta64(m, 'M')
                tct[j][m].append([LATS[mask], LONS[mask]])
#     for j in range(len(times)):
#         # print(j, end=',')
#         t, y, x = ts[j], ys[j], xs[j]
#         tct_grid[t, y-5:y+5, x-10:x+10] += 1
#     print(times[j], LATS[j], LONS[j])
# ds_tct.tct_grid[:] = tct_grid / 80
# ds_tct.to_netcdf('./data-20c/NOAA-20C_TCT_grid.nc')

In [ ]:
times[np.logical_and(times >= mix[0], times <= mix[0]+np.timedelta64(24))]

In [ ]:
time = pd.date_range('1900-01', '2015-12', freq='MS')
nt = len(time)
nx, ny = len(lon), len(lat)
ds_tct = xr.Dataset(
    {
        'tct_grid':(['time', 'lat', 'lon'], np.zeros((nt, ny, nx))),
    },
    coords={
        'time':(['time'], time),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
tct_grid = np.zeros((nt, ny, nx))

for i in range(1, 81):
    ds = xr.Dataset(
    {
        'tct_grid':(['time', 'lat', 'lon'], np.zeros((nt, ny, nx))),
    },
    coords={
        'time':(['time'], time),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
    )
    tct = np.zeros((nt, ny, nx))
    file_name = TCT_path+f'merged_0{i:02d}.txt'
    output_name = output_path+f'merged_0{i:02d}.nc'
    print(file_name, output_name)
    df = pd.read_csv(file_name, header=None, sep=',', names=['lon', 'lat', 'intensity', 'time'], on_bad_lines='skip').dropna()
    times = df.loc[:, 'time'].to_numpy(dtype='datetime64[M]')
    LATS = df.loc[:, 'lat'].to_numpy(dtype='int32')
    LONS = df.loc[:, 'lon'].to_numpy(dtype='int32')
    mask = times >= np.datetime64('1900-01')
    ts = (times - np.datetime64('1900-01')).astype(int)
    ys = LATS + 90
    xs = LONS
    for j in range(len(times)):
        # print(j, end=',')
        t, y, x = ts[j], ys[j], xs[j]
        tct[t, y-5:y+5, x-10:x+10] += 1
    # tct_grid[:] += tct
    ds.tct_grid[:] = tct
    ds.to_netcdf(output_name)
    # print(times[j], LATS[j], LONS[j])
# ds_tct.tct_grid[:] = tct_grid / 80
# ds_tct.to_netcdf('./data-20c/NOAA-20C_TCT_grid.nc')

In [ ]:
rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()
def process_space(ssta):
    ssta -= ssta.groupby('time.month').map(rolling)
    ssta_sum = ssta.sel(time=ssta.time.dt.month.isin([6, 7, 8, 9, 10, 11])).coarsen(time=6).mean()
    # ssta_win = ssta[11:-1].sel(time=ssta[11:-1].time.dt.month.isin([1, 2, 3, 4, 5, 12])).coarsen(time=6).mean()
    return ssta, ssta_sum

path_win = r'/Volumes/shiki/study/data/noaa-20c/NOAA/NOAA_20C_TCT_win/'
path_sum = r'/Volumes/shiki/study/data/noaa-20c/NOAA/NOAA_20C_TCT_sum/'
for i in range(1, 81):
    TCT_path_grid = output_path
    ds = xr.open_dataset(TCT_path_grid+f'merged_0{i:02d}.nc')
    tct = ds.tct_grid.loc[:, -40:40, 0:359]
    # tct, tct_sum, tct_win = process_space(tct)
    tct, tct_sum = process_space(tct)
    ds_sum = xr.Dataset({'tct_sum':tct_sum}, coords=tct_sum.coords)
    # ds_win = xr.Dataset({'tct_win':tct_win}, coords=tct_win.coords)
    ds_sum.to_netcdf(path_sum+f'merged_0{i:02d}.nc')
    # ds_win.to_netcdf(path_win+f'merged_0{i:02d}.nc')

In [ ]:
ds = xr.open_dataset(TCT_path_grid+f'merged_001.nc')
tct = ds.tct_grid.loc[:, -40:40, 0:359]
month = tct.time.dt.month
lat_nh = (tct.lat >= 5) & (tct.lat <= 25) 
lat_sh = (tct.lat >= -25) & (tct.lat <= -5)
is_jun_nov = (month >= 6) & (month <= 11)
is_dec_may = (month >= 12) | (month <= 5)
mask = (is_jun_nov & lat_nh) | (is_dec_may & lat_sh)

for i in range(1, 81):
    TCT_path_grid = output_path
    ds = xr.open_dataset(TCT_path_grid+f'merged_0{i:02d}.nc')
    tct = ds.tct_grid.loc[:, -40:40, 0:359]
    tct = tct.groupby('time.month') - tct.groupby('time.month').mean()
    tct_hov = tct.where(mask).mean('lat')
    tct_hov = tct_hov.assign_coords(
                year=tct_hov.time.dt.year,
                month=tct_hov.time.dt.month
            ).set_index(time=["year", "month"]).unstack("time").transpose('year', 'month', 'lon')
    ds_hov = xr.Dataset({'tct_hov':tct_hov}, coords=tct_hov.coords)
    output_name = f'/Volumes/shiki/study/data/noaa-20c/NOAA/NOAA_20C_TCT_hov/merged_0{i:02d}.nc'
    print(output_name)
    ds_hov.to_netcdf(output_name)

In [ ]:
ds_hov.tct_hov[:, 6].plot()

In [ ]:
time = pd.date_range('1806-01', '2015-12', freq='MS')
nt = len(time)
nx, ny = len(lon), len(lat)
ds_tc = xr.Dataset(
    {
        'tcg_grid':(['time', 'lat', 'lon'], np.zeros((nt, ny, nx))),
        'tcg_current':(['basin', 'time'], np.zeros((6, nt))),
        'tcg_series':(['basin', 'time'], np.zeros((6, nt))),
        'tcg_series_max':(['basin', 'time'], np.zeros((6, nt))),
        'tcg_series_min':(['basin', 'time'], np.zeros((6, nt))),
    },
    coords={
        'basin':(['basin'], ['SH', 'NS', 'NI', 'WNP', 'ENP', 'NA']),
        'time':(['time'], time),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
ds_tc.tcg_series_min[:] = 1000
max_tc = np.zeros((6, nt))
min_tc = np.zeros_like(max_tc)
tcg_current = np.zeros((6, nt))
tcg_grid = np.zeros((nt, ny, nx))
for i in range(1, 81):
    print(i)
    loc = np.loadtxt(TC_path1d+f'Genesis_0{i:02d}.txt', usecols=[0, 1], delimiter=',')
    times = np.loadtxt(TC_path1d+f'Genesis_0{i:02d}.txt', usecols=[-1], delimiter=',', dtype='datetime64[M]')
    lons = loc[:, 0].astype(int)
    lats = loc[:, 1].astype(int) + 90
    nts = (times - np.datetime64('1806-01')).astype(int)
    for j in range(len(times)):
        # print(times[j])
        LON, LAT = lons[j], lats[j]
        if LAT-90 < 0:
            basin = 0
        else:
            if LON <= 100:
                basin = 2
            elif LON <= 180:
                basin = 3
            else:
                LON_ = -2.15*(LAT-90) + 298.05
                if LON <= LON_:
                    basin = 4
                else:
                    basin = 5
        # ds_tc.tcg_current.loc[basin, times[j]] += 1
        # ds_tc.tcg_grid.loc[times[j], LAT-5:LAT+5, LON-10:LON+10] += 1
        tcg_grid[nts[j], LAT-5:LAT+5, LON-10:LON+10] += 1
        tcg_current[basin, nts[j]] += 1

    ds_tc.tcg_current[:] = tcg_current
    ds_tc.tcg_series[:] += ds_tc.tcg_current[:]
    mask = ds_tc.tcg_current.values <= ds_tc.tcg_series_max.values
    ds_tc.tcg_series_max[:] = ds_tc.tcg_series_max.where(mask, ds_tc.tcg_current)
    mask = ds_tc.tcg_current.values >= ds_tc.tcg_series_min.values
    ds_tc.tcg_series_min[:] = ds_tc.tcg_series_min.where(mask, ds_tc.tcg_current)
    tcg_current[:] = 0
    
ds_tc.tcg_grid[:] = tcg_grid/80
ds_tc.tcg_series[:] /= 80

ds_tc.to_netcdf('./data-20c/NOAA-20C_TC_grid.nc')

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
ds_tc = xr.open_dataset('./data-20c/NOAA-20C_TC_grid.nc').sel(time=slice('1900-01', None))
basin = [3, 4, 5, 0]
series = ds_tc.tcg_series[basin, :].coarsen(time=12).sum()[:, :-1]
series_max = ds_tc.tcg_series_max[basin, :].coarsen(time=12).sum()[:, :-1]
series_min = ds_tc.tcg_series_min[basin, :].coarsen(time=12).sum()[:, :-1]
time = series.time.dt.year
titles = series.basin.values
ylim = [[0, 40], [0, 30], [0, 30], [0, 40]]
for i, ax in enumerate(axs.flatten()):
    ax.plot(time, series[i], linewidth=1.3, color='sienna')
    ax.fill_between(time, series_max[i], series_min[i], color='#EAF2DA')
    ax.set_ylim(ylim[i])
    ax.set_title(titles[i], fontsize=12)
fig.savefig('./pic-20c/NOAA-20C_TC_series_org.png', dpi=300)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 5))
series = ds_tc.tcg_series[[3, 4, 5], :].coarsen(time=12).sum()[:, :-1]
series_max = ds_tc.tcg_series_max[[3, 4, 5], :].coarsen(time=12).sum()[:, :-1]
series_min = ds_tc.tcg_series_min[[3, 4, 5], :].coarsen(time=12).sum()[:, :-1]

series -= series.rolling(time=30, min_periods=1,center=True).mean()
series_max -= series_max.rolling(time=30, min_periods=1,center=True).mean()
series_min -= series_min.rolling(time=30, min_periods=1,center=True).mean()
time = series.time.dt.year
titles = ['WNP', 'ENP', 'NA']
ylim = [[-20, 20], [-20, 20], [-20, 20]]
for i, ax in enumerate(axs.flatten()):
    ax.plot(time, series[i], linewidth=1.3, color='sienna')
    ax.fill_between(time, series_max[i], series_min[i], color='#EAF2DA')
    ax.set_ylim(ylim[i])
    ax.set_title(titles[i], fontsize=12)
# fig.savefig('./pic-20c/NOAA-20C_TC_series.png', dpi=300)